In [3]:
import torch
import sklearn
import numpy as np


In [ ]:
torch.rand(5)
torch.randn(5)
torch.empty(5)
torch.zeros(5)
torch.ones(5)


tensor([1., 1., 1., 1., 1.])

In [ ]:
torch.full((5,5),8)

tensor([[8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8]])

In [ ]:
torch.arange(0,10)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
torch.linspace(0,10,steps=5)

tensor([ 0.0000,  2.5000,  5.0000,  7.5000, 10.0000])

In [ ]:
houses = torch.tensor([
    [2,65,15,285],
    [3,95,8,425],
    [4,88,42,380],
    [5,88,42,295],
    [2,58,58,245],
] , dtype = torch.float32)

In [ ]:
features = houses[:,:-1]
targets = houses[:,-1]
print(features)
print(targets)



tensor([[ 2., 65., 15.],
        [ 3., 95.,  8.],
        [ 4., 88., 42.],
        [ 5., 88., 42.],
        [ 2., 58., 58.]])
tensor([285., 425., 380., 295., 245.])


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 1. DATASET — always inherit from Dataset, implement __len__ and __getitem__
class MyDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

# 2. MODEL — always inherit from nn.Module, define layers in __init__, wire them in forward
class MyModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    
    def forward(self, x):
        return self.net(x)

# 3. LOSS + 4. OPTIMIZER
model = MyModel(784, 256, 10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 5. TRAINING LOOP — this exact pattern, every single time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



for epoch in range(num_epochs):
    model.train()  # sets dropout/batchnorm to training mode
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        logits = model(batch_x)          # forward pass
        loss = criterion(logits, batch_y) # compute loss
        
        optimizer.zero_grad()             # clear old gradients
        loss.backward()                   # compute new gradients
        optimizer.step()                  # update parameters
    
    # Evaluation — always use torch.no_grad() and model.eval()
    model.eval()
    with torch.no_grad():
        # compute validation metrics here
        pass

NameError: name 'train_loader' is not defined

In [1]:
"""
THE UNIVERSAL PYTORCH TRAINING TEMPLATE
========================================
This exact structure is used in EVERY PyTorch project — from a 2-layer MLP
to GPT-4. Memorize the 5 components: Data → Model → Loss → Optimizer → Loop.

Task: Classify 2D points into 2 classes (inside vs outside a circle).
Simple enough to visualize, complex enough to need a neural network.
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

# ============================================================
# COMPONENT 1: DATA
# Rule: Always make a Dataset class. Always use DataLoader.
# Dataset = WHERE your data lives
# DataLoader = HOW it's served (batching, shuffling)
# ============================================================

class CircleDataset(Dataset):
    """
    Points in 2D space. Label 1 if inside unit circle, 0 if outside.
    This is our "Hello World" — simple enough to visualize in your head.
    """
    def __init__(self, n_samples=1000):
        # Generate random 2D points between -2 and 2
        self.data = torch.randn(n_samples, 2)  # shape: (1000, 2)
        
        # Label: 1 if x²+y² < 1 (inside circle), else 0
        distances = self.data[:, 0]**2 + self.data[:, 1]**2
        self.labels = (distances < 1.0).long()  # shape: (1000,)
        
        print(f"Dataset: {n_samples} points, "
              f"{self.labels.sum().item()} inside circle, "
              f"{n_samples - self.labels.sum().item()} outside")
    
    def __len__(self):
        # DataLoader calls this to know total size
        return len(self.data)
    
    def __getitem__(self, idx):
        # DataLoader calls this to get one sample
        # ALWAYS return (input, label) tuple
        return self.data[idx], self.labels[idx]


# Create datasets
train_dataset = CircleDataset(n_samples=1000)
test_dataset = CircleDataset(n_samples=200)

# DataLoader wraps dataset — handles batching + shuffling
# These 3 args matter most: dataset, batch_size, shuffle
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)  # no shuffle for eval

# Quick sanity check — ALWAYS do this
batch_x, batch_y = next(iter(train_loader))
print(f"\nBatch shapes: x={batch_x.shape}, y={batch_y.shape}")
# Expected: x=torch.Size([32, 2]), y=torch.Size([32])


# ============================================================
# COMPONENT 2: MODEL
# Rule: Inherit nn.Module. Define layers in __init__.
# Wire them in forward(). That's it. Nothing else.
# ============================================================

class SimpleClassifier(nn.Module):
    """
    2-layer MLP. The simplest possible neural network.
    Input(2) → Hidden(32) → ReLU → Hidden(16) → ReLU → Output(2)
    """
    def __init__(self, input_dim=2, hidden_dim=32, num_classes=2):
        super().__init__()  # NEVER forget this line
        
        # nn.Sequential = layers in order, no branching
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # 2 → 32
            nn.ReLU(),                           # activation
            nn.Linear(hidden_dim, hidden_dim // 2),  # 32 → 16
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes),  # 16 → 2 (raw logits)
        )
        # NOTE: No softmax here! CrossEntropyLoss includes it.
        # This is a common interview question.
    
    def forward(self, x):
        # x shape: (batch_size, 2)
        return self.network(x)  # output shape: (batch_size, 2)


# Instantiate model
model = SimpleClassifier()

# Print parameter count — good habit, interviewers like this
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {total_params} parameters")
print(model)  # shows architecture


# ============================================================
# COMPONENT 3 & 4: LOSS + OPTIMIZER
# Loss = "how wrong am I?" (GPS distance)
# Optimizer = "how do I fix it?" (GPS directions)
# ============================================================

criterion = nn.CrossEntropyLoss()  # classification → CrossEntropy
# For regression: nn.MSELoss()
# For binary/multi-label: nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Adam is the safe default. SGD + momentum for when you need control.
# lr=1e-3 is the safe starting point for Adam.


# ============================================================
# COMPONENT 5: THE TRAINING LOOP
# This is THE pattern. Forward → Loss → Zero → Backward → Step.
# Mnemonic: "Feed, Measure, Clean, Blame, Fix"
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"\nTraining on: {device}")

num_epochs = 20

for epoch in range(num_epochs):
    # ---- TRAINING PHASE ----
    model.train()  # enable dropout/batchnorm training behavior
    train_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        # Move data to device (GPU if available)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # THE 5 SACRED LINES — same in every project
        logits = model(batch_x)              # 1. FEED (forward pass)
        loss = criterion(logits, batch_y)     # 2. MEASURE (compute loss)
        optimizer.zero_grad()                 # 3. CLEAN (clear old gradients)
        loss.backward()                       # 4. BLAME (backprop)
        optimizer.step()                      # 5. FIX (update weights)
        
        # Track metrics
        train_loss += loss.item() * batch_x.size(0)
        _, predicted = logits.max(1)  # get class with highest logit
        correct += predicted.eq(batch_y).sum().item()
        total += batch_y.size(0)
    
    train_loss /= total
    train_acc = 100.0 * correct / total
    
    # ---- EVALUATION PHASE ----
    model.eval()  # disable dropout/batchnorm → use running stats
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # CRITICAL: saves memory, speeds up eval
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            test_loss += loss.item() * batch_x.size(0)
            _, predicted = logits.max(1)
            correct += predicted.eq(batch_y).sum().item()
            total += batch_y.size(0)
    
    test_loss /= total
    test_acc = 100.0 * correct / total
    
    # Print every 5 epochs to keep output clean
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.1f}%")


# ============================================================
# BONUS: INFERENCE — how to use the trained model
# ============================================================

model.eval()
with torch.no_grad():
    # Test with a known point: (0, 0) is inside the circle
    test_point = torch.tensor([[0.0, 0.0]]).to(device)
    logits = model(test_point)
    probs = torch.softmax(logits, dim=1)  # NOW we use softmax (only at inference!)
    pred_class = logits.argmax(dim=1).item()
    confidence = probs[0][pred_class].item()
    print(f"\nPoint (0,0) → Class {pred_class} (inside={pred_class==1}) "
          f"with {confidence:.1%} confidence")
    
    # Test with a point outside: (1.5, 1.5) → distance = 4.5 > 1
    test_point = torch.tensor([[1.5, 1.5]]).to(device)
    logits = model(test_point)
    probs = torch.softmax(logits, dim=1)
    pred_class = logits.argmax(dim=1).item()
    confidence = probs[0][pred_class].item()
    print(f"Point (1.5,1.5) → Class {pred_class} (inside={pred_class==1}) "
          f"with {confidence:.1%} confidence")

print("\n✓ Training complete. This exact pattern works for MNIST, ImageNet, ViT — everything.")

Dataset: 1000 points, 376 inside circle, 624 outside
Dataset: 200 points, 77 inside circle, 123 outside

Batch shapes: x=torch.Size([32, 2]), y=torch.Size([32])

Model: 658 parameters
SimpleClassifier(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=2, bias=True)
  )
)

Training on: cpu
Epoch [ 5/20] Train Loss: 0.3467 | Train Acc: 89.7% | Test Loss: 0.2890 | Test Acc: 96.5%
Epoch [10/20] Train Loss: 0.1415 | Train Acc: 98.0% | Test Loss: 0.1153 | Test Acc: 98.5%
Epoch [15/20] Train Loss: 0.0942 | Train Acc: 98.3% | Test Loss: 0.0757 | Test Acc: 99.0%
Epoch [20/20] Train Loss: 0.0733 | Train Acc: 98.6% | Test Loss: 0.0588 | Test Acc: 99.5%

Point (0,0) → Class 1 (inside=True) with 99.8% confidence
Point (1.5,1.5) → Class 0 (inside=False) with 100.0% confidence

✓ Training complete. This exact pattern works for MNIST

In [ ]:
w = torch.rand(size = (2 , 1) , requires_grad= True  )

In [11]:
x = torch.randn(5)
x

tensor([-0.0309,  0.0764,  0.3433,  0.2054, -0.4186])